# Peak calling for specific cases

Using the latest on September 29th, 2025

** ALSO, will include more normal cells (CN > 3) instead of cutting off at 5 to capture 
low concentrating ones **

Run the peak calling algorithm, annotate the peaks, save the number peaks, then plot the peaks. 


1. Load the list of data
2. For each data:
    1. load the data
    2. filter for individual-gene from the summary file
    3. in each file, filter cells to only tumors from the qc files
    4. run the peak caller
    5. save the results
    6. plot the annotated peaks
5. Gather the results in one place


Use the same file that is used to run the main mixture model pipeline.

In [ ]:
from peak_detection_utils import *

# Set font to Arial
force_arial()
rcParams['font.family'] = 'Arial'
sc.settings.set_figure_params(dpi_save=300)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


# Setup Original Oncogene CNV files
oncogene_files = glob('/data1/shahs3/users/leej39/dlp/analysis/normalization/oncogene_cn_final//*.sv-seg-cn.oncogene.cn.csv')

# Setup the output directories
outDir = '/data1/shahs3/users/salehis/ecdna/results/peak_detection_november_23_2025'

# Load the ecDNA mixture model results
oncogenes = pd.read_csv('/data1/shahs3/users/salehis/ecdna/results/model_embedding_november_07_2025_ONCO_genes/tables/processed_model_results_nov_07_2025.csv')



In [ ]:
# Minimum process
figuresDir, tablesDir, dataDir = setup_dirs(outDir)

oncogenes['individual'] = oncogenes['system_name'].tolist()
oncogenes['gene'] = oncogenes['locus'].tolist()
oncogenes['id'] = oncogenes['individual'].astype(str) + '__' + oncogenes['gene'].astype(str)


In [ ]:
rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/peak_detection_**/figures/*.p* \
    /Users/salehis/Projects/ecdna

In [ ]:
ignore_qc = True
do_plot = False
results = None

minimal_normal = 3 ## New param, include more normal cells with CN > 3

missed = []
MAX_i = 10

for i, item in enumerate(oncogenes.iterrows()):
    item = item[1]
    # if i > MAX_i:
    #     print("Breaking after 10 iterations for testing")
    #     break
    individual = item["individual"]
    # Extract the cn path and qc path from oncogene_files
    try:
        fpath, qc_path = get_paths(
            individual=individual, ignore_qc=ignore_qc, oncogene_files=oncogene_files
        )
        locus = item["gene"]
        if ignore_qc:
            tumor_cells = None
        else:
            tumor_cells = get_tumor_cells(qc_path=qc_path)
        print(f"Processing {fpath} for locus {locus}")
        dataset_name, locus, n_peaks, pickle_path = handle_item(
            fpath=fpath,
            locus=locus,
            verbose=False,
            do_plot=do_plot,
            tumor_cells=tumor_cells,
            figuresDir=figuresDir,
            dataDir=dataDir,
            drop_normals=True,
            minimal_normal=minimal_normal,
        )
        # Create a tmp data.frame to add to the results
        tmp_df = pd.DataFrame(
            {
                "dataset_name": [dataset_name],
                "locus": [locus],
                "n_peaks": [n_peaks],
                "pickle_path": [pickle_path],
            }
        )
        if results is None:
            results = tmp_df
        else:
            results = pd.concat([results, tmp_df], ignore_index=True)
        # print(f"Processed {i+1}/{len(results)}: {dataset_name}, {locus}, n_peaks: {n_peaks}, saved to {pickle_path}")
    except Exception as e:
        print(f"Error processing {individual}, {locus}: {e}")
        missed.append((individual, locus, str(e)))


In [ ]:

# sort by number of peaks
results = results.sort_values(by='n_peaks', ascending=False)
# save as tsv in tables
results_save_path = os.path.join(tablesDir, 'peak_detection_results.tsv')
results.to_csv(results_save_path, sep='\t', index=False)
print(f"Saved peak detection results to:\n{results_save_path}")



## Remove the ecDNA cases

In [ ]:
ecdna = pd.read_csv('/data1/shahs3/users/salehis/ecdna/results/model_embedding_november_07_2025_ONCO_genes/tables/processed_model_results_nov_07_2025.csv')
# Pick score > .5 
ecdna = ecdna[ecdna['score'] > 0.4]


results_IC = results.copy()
# Remove ecDNA cases from results_IC
results_IC['id'] = results_IC['dataset_name'].astype(str) + '_' + results_IC['locus'].astype(str)
results_IC = results_IC[~results_IC['id'].isin(ecdna['id'].tolist())]


# Plot the distribution of n_peaks
results_IC['n_peaks'].value_counts().sort_index()
# PLot
plt.figure(figsize=(4,3))
sns.countplot(data=results_IC, x='n_peaks', order=sorted(results_IC['n_peaks'].unique()))
plt.xlabel('Number of Peaks Detected')
plt.ylabel('Number of Samples')
plt.title('Distribution of Detected Peaks (Non-ecDNA Samples)')
plt.savefig(os.path.join(figuresDir, 'n_peaks_distribution_non_ecdna_samples.png'), dpi=300, bbox_inches='tight')
plt.close()



In [ ]:

# Find zero peak cases
zero_peak_comb = results_IC[results_IC['n_peaks'] == 0].copy()
zero_peak_comb['dataset_name'].value_counts().shape

zero_peak_comb_new = zero_peak_comb.copy()
# drop pickle path
zero_peak_comb_new = zero_peak_comb_new.drop(columns=['pickle_path'])

zero_peak_comb_new['std_cn'] = np.nan
zero_peak_comb_new['max_cn'] = np.nan
zero_peak_comb_new['n_cells'] = np.nan

tmp = None
# annotate these cases by the number of cells and st and max copy number
for idx, row in zero_peak_comb.iterrows():
    id_ = row['dataset_name']
    individual = row['dataset_name']
    locus = row['locus']
    # load the data
    fpath = get_paths(individual=individual, ignore_qc=True, oncogene_files=oncogene_files)[0]
    vals, _, _ = load_data(
        dat_path=fpath,
        locus=locus,
        do_log=False,
        do_prune_left=False,
        keep_cells=tumor_cells,
    )
    # compute the standard deviation
    std_vals = np.std(vals)
    max_cn = np.max(vals)
    n_cells = len(vals)
    # Add these as new columns to the zero_peak_comb_new
    # (1) build a parallel data.frame, add the new values to, then concatenate at the vary end
    zero_peak_comb_new.loc[idx, 'std_cn'] = std_vals
    zero_peak_comb_new.loc[idx, 'max_cn'] = max_cn
    zero_peak_comb_new.loc[idx, 'n_cells'] = n_cells

# now concatenate the columns
# sort by the number of cells
zero_peak_comb_new = zero_peak_comb_new.sort_values(by='n_cells', ascending=False)

# save this to tables
zero_peak_save_path = os.path.join(tablesDir, 'zero_peak_detection_cases_non_ecdna_samples.tsv')
zero_peak_comb_new.to_csv(zero_peak_save_path, sep='\t', index=False)
    

In [ ]:
results = pd.read_csv(os.path.join(tablesDir, 'peak_detection_results.tsv'), sep='\t')

In [ ]:
# Load the pickle files, and add a row for each dataset, locus, peak, left_base_position, right_base_position, peak_height
def load_peak_results(pickle_path):
    with open(pickle_path, 'rb') as f:
        results_dict = pickle.load(f)
    return results_dict

detailed_results = None
adding = 0
for i, items in enumerate(results.iterrows()):
    item = items[1]
    pickle_path = item['pickle_path']
    results_dict = load_peak_results(pickle_path)
    if results_dict is None or len(results_dict['left_base_positions']) == 0:
        print(f"Skipping {pickle_path} as results_dict is None")
        continue
    # Add the results to the results data.frame
    peak_id = 0
    for j in range(len(results_dict['left_base_positions'])):
        left_base_position = results_dict['left_base_positions'][j]
        right_base_position = results_dict['right_base_positions'][j]
        peak_height = results_dict['peak_heights_list'][j]
        new_row = {
            'dataset_name': results_dict['dataset_name'],
            'locus': results_dict['locus'],
            'n_peaks': len(results_dict['left_base_positions']),
            'left_base_position': left_base_position,
            'right_base_position': right_base_position,
            'peak_height': peak_height,
            'peak_width': right_base_position - left_base_position,
            'peak_id': peak_id,
            'n_cells': results_dict['n_cells'],
        }
        peak_id += 1
        if detailed_results is None:
            detailed_results = pd.DataFrame([new_row])
        else:
            detailed_results = pd.concat([detailed_results, pd.DataFrame([new_row])], ignore_index=True)
            adding += 1

# Save
detailed_results.to_csv(os.path.join(tablesDir, 'peak_detection_detailed_results.tsv'), sep='\t', index=False)

# sort ascending by peak_height
detailed_results = detailed_results.sort_values(by='peak_height', ascending=True)
detailed_results

In [ ]:
detailed_results = pd.read_csv(os.path.join(tablesDir, 'peak_detection_detailed_results.tsv'), sep='\t')

In [ ]:
detailed_results.sort_values(by='peak_width', ascending=False)

In [ ]:
detailed_results['peak_height_fraction'] = detailed_results['peak_height'] / detailed_results['n_cells']
# Order by peak_height_fraction
detailed_results = detailed_results.sort_values(by='peak_height_fraction', ascending=True)

## Find datasets that have zero peaks entirely


Fig2B is created as follows:

1. Read dataset by locus table
2. Subset to rows where n_peaks == 0
3. Count the number of unique datasets


In [ ]:
# Load the results again 
results = pd.read_csv(os.path.join(tablesDir, 'peak_detection_results.tsv'), sep='\t')

# Find datasets where there is no peaks at any locus at all
# (1) for each dataset, how many zero peak loci are there?

# Fig2B is showing sample__locus pairs
zero_peak_comb = results[results['n_peaks'] == 0].copy()
zero_peak_comb['id'] = zero_peak_comb['dataset_name'].astype(str) + '_' + zero_peak_comb['locus'].astype(str)

# Load ecDNA cases and exclude them
ecdna = pd.read_csv('/data1/shahs3/users/salehis/ecdna/results/model_embedding_november_07_2025_ONCO_genes/tables/processed_model_results_nov_07_2025.csv')
# Pick score > .5 
ecdna = ecdna[ecdna['score'] > 0.4]


ecdna[ecdna['system_name'] == 'NCI-H524']


zero_peak_comb = zero_peak_comb[~zero_peak_comb['id'].isin(ecdna['id'].tolist())]
zero_peak_comb['dataset_name'].value_counts().shape

# Check os of these


## For the ones without peaks, plot in order to debug

rsync -azvp --relative \
    iris:/data1/shahs3/users/salehis/ecdna/./results/peak_detection_november_08_2025/figures_debug_zero_peaks/*.p* \
    /Users/salehis/Projects/ecdna

In [ ]:
ignore_qc = True
do_plot = True
results = None

missed = []

#tmp_dataDir = os.path.join(outDir, 'data_debug_zero_peaks')
#tmp_figuresDir = os.path.join(outDir, 'figures_debug_zero_peaks')
tmp_dataDir = os.path.join(outDir, 'data_debug_zero_peaks_min3')
tmp_figuresDir = os.path.join(outDir, 'figures_debug_zero_peaks_min3')

zero_peak_comb[zero_peak_comb['dataset_name'] == 'SPECTRUM-OV-009']

# Create these dirs
os.makedirs(tmp_dataDir, exist_ok=True)
os.makedirs(tmp_figuresDir, exist_ok=True)

for i, item in enumerate(oncogenes.iterrows()):
    item = item[1]
    individual = item["individual"]
    # Extract the cn path and qc path from oncogene_files
    try:
        fpath, qc_path = get_paths(
            individual=individual, ignore_qc=ignore_qc, oncogene_files=oncogene_files
        )
        locus = item["gene"]
        # Check only those in zero_peak_comb
        id_ = f"{individual}_{locus}"
        if id_ not in zero_peak_comb_new["id"].tolist():
            continue
        #if locus != 'KRAS' or individual != 'SPECTRUM-OV-009':
        # if locus != 'KRAS' or individual != 'SA604':
        #     continue
        else:
            print(f"Processing {id_}")
        if ignore_qc:
            tumor_cells = None
        else:
            tumor_cells = get_tumor_cells(qc_path=qc_path)
        print(f"Processing {fpath} for locus {locus}")
        dataset_name, locus, n_peaks, pickle_path = handle_item(
            fpath=fpath,
            locus=locus,
            verbose=False,
            do_plot=do_plot,
            tumor_cells=tumor_cells,
            figuresDir=tmp_figuresDir,
            dataDir=tmp_dataDir,
            drop_normals=True,
            minimal_normal=3,
        )
        print(f"found {n_peaks} peaks for {id_}")
        assert individual == dataset_name, f"Mismatch: {individual} vs {dataset_name}"
        # Create a tmp data.frame to add to the results
        tmp_df = pd.DataFrame(
            {
                "dataset_name": [dataset_name],
                "locus": [locus],
                "n_peaks": [n_peaks],
                "pickle_path": [pickle_path],
            }
        )
        if results is None:
            results = tmp_df
        else:
            results = pd.concat([results, tmp_df], ignore_index=True)
        # print(f"Processed {i+1}/{len(results)}: {dataset_name}, {locus}, n_peaks: {n_peaks}, saved to {pickle_path}")
    except Exception as e:
        print(f"Error processing {individual}, {locus}: {e}")
        missed.append((individual, locus, str(e)))


# How many are zeros now?
zero_results = results[results['n_peaks'] == 0].copy()


# for each pickle file, load, and add the number of cells
for i, items in enumerate(results.iterrows()):
    item = items[1]
    pickle_path = item['pickle_path']
    try:
        results_dict = load_peak_results(pickle_path)
        n_cells = results_dict['n_cells']
        results.at[i, 'n_cells'] = n_cells
    except Exception as e:
        print(f"Error loading pickle {pickle_path}: {e}")


# plot a histogram of n_cells
plt.figure(figsize=(6,4))
sns.histplot(results['n_cells'], bins=30, kde=False)
plt.xlabel('Number of cells in sample')
plt.ylabel('Number of samples')
plt.title('Distribution of number of cells in samples with zero detected peaks')
plt.tight_layout()
plt.savefig(os.path.join(outDir, 'figures_debug_zero_peaks', 'histogram_n_cells_zero_peaks.png'))
plt.show()

# what is the mean and median and the quantiles of n_cells
n_cells_series = results['n_cells'].dropna()
mean_n_cells = n_cells_series.mean()
median_n_cells = n_cells_series.median()
quantiles_n_cells = n_cells_series.quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99])



## Inspect individual cells



In [ ]:
individual = 'SPECTRUM-OV-081'
locus_name = 'KLF5'
fpath, qc_path = get_paths(
    individual=individual, ignore_qc=ignore_qc, oncogene_files=oncogene_files
)
vals, _, _ = load_data(dat_path=fpath, locus=locus_name, do_log=False, do_prune_left=False, keep_cells=None)

In [ ]:
# overall, what is their standard deviation?
ignore_qc = True
do_plot = False
results = None

missed = []

tmp_dataDir = os.path.join(outDir, "data_debug_zero_peaks")
tmp_figuresDir = os.path.join(outDir, "figures_debug_zero_peaks")

# Create these dirs
os.makedirs(tmp_dataDir, exist_ok=True)
os.makedirs(tmp_figuresDir, exist_ok=True)

for i, item in enumerate(oncogenes.iterrows()):
    item = item[1]
    individual = item["individual"]
    # Extract the cn path and qc path from oncogene_files
    try:
        fpath, qc_path = get_paths(
            individual=individual, ignore_qc=ignore_qc, oncogene_files=oncogene_files
        )
        locus = item["gene"]
        # Check only those in zero_peak_comb
        id_ = f"{individual}_{locus}"
        if id_ not in zero_peak_comb["id"].tolist():
            continue
        if ignore_qc:
            tumor_cells = None
        else:
            tumor_cells = get_tumor_cells(qc_path=qc_path)
        print(f"Processing {fpath} for locus {locus}")
        # load the data
        vals, _, _ = load_data(
            dat_path=fpath,
            locus=locus,
            do_log=False,
            do_prune_left=False,
            keep_cells=tumor_cells,
        )
        # compute the standard deviation
        std_vals = np.std(vals)
        max_cn = np.max(vals)
        n_cells = len(vals)
        # Create a tmp data.frame to add to the results
        # if individual == 'SPECTRUM-OV-009' and locus == 'FOXA1':
        #     breakpoint()
        tmp_df = pd.DataFrame(
            {
                "dataset_name": [individual],
                "locus": [locus],
                "std_vals": [std_vals],
                "max_cn": [max_cn],
                "n_cells": [n_cells],
            }
        )
        if results is None:
            results = tmp_df
        else:
            results = pd.concat([results, tmp_df], ignore_index=True)
        # print(f"Processed {i+1}/{len(results)}: {individual}, {locus}, n_peaks: {n_peaks}, saved to {pickle_path}")
    except Exception as e:
        print(f"Error processing {individual}, {locus}: {e}")
        missed.append((individual, locus, str(e)))


# is FOXA2 in this
results[results["locus"] == "FOXA1"]
# is SPECTRUM-OV-009 in zero_peak_comb?
zero_peak_comb[zero_peak_comb["dataset_name"] == "SPECTRUM-OV-009"]

results[results["dataset_name"] == "SPECTRUM-OV-036"]
results[results["dataset_name"] == "SA604"]


# describe results
results.describe()
# compute a lot of quantiles
results["std_vals"].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

# show the ones that are more than 3
results[results["std_vals"] > 3.0]

In [ ]:




# for each dataset, just some over n_peaks

zero_peak_datasets = results[['dataset_name', 'n_peaks']].groupby('dataset_name').sum()
# filter down to only those with zero peaks
zero_peak_datasets = zero_peak_datasets[zero_peak_datasets['n_peaks'] == 0]
zero_peak_datasets = zero_peak_datasets.reset_index()


